# 4. RAG + LLM 비만 관리 가이드라인 추천

**파이프라인:**
1. `data/guidelines/` 내 TXT/PDF 파일을 문장 단위로 청크 분할
2. SentenceTransformer로 청크 임베딩 생성 후 `artifacts/rag_chunks.npz`에 저장
3. 클러스터 요약문(`artifacts/cluster_summaries.json`)을 쿼리로 사용
4. 코사인 유사도로 관련 청크 Top-k 검색
5. 검색된 청크 + 클러스터 프로파일을 Gemini에 전달 → 맞춤 추천 생성
6. BERTScore로 답변 품질 평가

**입력:** `data/guidelines/*.txt|.pdf`, `artifacts/cluster_summaries.json`
**환경 변수:** `.env` 파일에 `GEMINI_API_KEY` 설정 필요

In [ ]:
%pip install -q sentence-transformers PyPDF2 nltk bert-score google-generativeai python-dotenv

In [ ]:
import os, re, json, glob
import numpy as np
import PyPDF2
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from dotenv import load_dotenv
import google.generativeai as genai

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

load_dotenv('../.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found. Create .env from .env.example and set your key.')

genai.configure(api_key=GEMINI_API_KEY)
ARTIFACTS = '../artifacts'
GUIDELINES_DIR = '../data/guidelines'

## 가이드라인 문서 로드 및 청크 분할

문장 토크나이저로 분할한 후 window=3, stride=2로 슬라이딩 윈도우 청크를 만들면
각 청크에 문맥이 충분히 포함됩니다.

In [ ]:
def load_text_from_file(path: str) -> str:
    if path.endswith('.pdf'):
        text = []
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text.append(t)
        return ' '.join(text)
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()


def make_chunks(text: str, window: int = 3, stride: int = 2) -> list[str]:
    """Sliding-window sentence-level chunking."""
    sentences = [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 20]
    chunks = []
    for i in range(0, len(sentences), stride):
        chunk = ' '.join(sentences[i:i + window])
        if chunk:
            chunks.append(chunk)
    return chunks


all_chunks, all_sources = [], []
files = glob.glob(f'{GUIDELINES_DIR}/*.txt') + glob.glob(f'{GUIDELINES_DIR}/*.pdf')

if not files:
    raise FileNotFoundError(f'No guideline files found in {GUIDELINES_DIR}')

for fp in files:
    doc_name = os.path.basename(fp)
    text = load_text_from_file(fp)
    chunks = make_chunks(text)
    all_chunks.extend(chunks)
    all_sources.extend([doc_name] * len(chunks))
    print(f'  {doc_name}: {len(chunks)} chunks')

print(f'\n총 청크 수: {len(all_chunks)}')
print('\n샘플 청크 (처음 3개):')
for i, c in enumerate(all_chunks[:3]):
    print(f'  [{i}] {c[:150]}...')

## 청크 임베딩 생성 및 저장

In [ ]:
print('SentenceTransformer 로딩 중...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'임베딩 생성 중... ({len(all_chunks)}개 청크)')
chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # 코사인 유사도 계산을 내적으로 가속
)

np.savez(
    f'{ARTIFACTS}/rag_chunks.npz',
    embeddings=chunk_embeddings,
    chunks=np.array(all_chunks),
    sources=np.array(all_sources)
)
print(f'rag_chunks.npz 저장 완료: {chunk_embeddings.shape}')

## 클러스터 요약문 로드

In [ ]:
with open(f'{ARTIFACTS}/cluster_summaries.json', encoding='utf-8') as f:
    cluster_summaries = json.load(f)

print(f'{len(cluster_summaries)}개 클러스터 요약문 로드 완료')
for s in cluster_summaries[:2]:
    print(f"  Cluster {s['cluster']}: {s['summary'][:100]}...")

## RAG 검색 함수

In [ ]:
def retrieve_chunks(query: str, top_k: int = 5) -> list[dict]:
    """
    Returns top-k guideline chunks most relevant to the query.
    Assumes rag_chunks.npz is pre-loaded into module scope.
    """
    data_npz = np.load(f'{ARTIFACTS}/rag_chunks.npz', allow_pickle=True)
    embs = data_npz['embeddings']                  # (N, 384)
    texts = data_npz['chunks'].tolist()
    sources = data_npz['sources'].tolist()

    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]
    scores = embs @ q_emb                          # dot product = cosine (normalized)
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [
        {'chunk': texts[i], 'source': sources[i], 'score': float(scores[i])}
        for i in top_idx
    ]

## 추천 생성 함수 (Gemini)

In [ ]:
def generate_recommendation(cluster_summary: str, top_k: int = 5) -> dict:
    """Retrieve relevant guideline chunks and generate LLM recommendations."""
    retrieved = retrieve_chunks(cluster_summary, top_k=top_k)

    context = '\n\n'.join(
        f'[Source: {r["source"]}] {r["chunk"]}' for r in retrieved
    )

    prompt = (
        'You are an expert obesity management clinician.\n'
        'Based on the patient profile and clinical guidelines below, '
        'provide specific, actionable recommendations.\n\n'
        f'## Patient Group Profile\n{cluster_summary}\n\n'
        f'## Relevant Guideline Excerpts\n{context}\n\n'
        '## Instructions\n'
        '1. Summarize key health risks for this group.\n'
        '2. Give 3-5 specific dietary recommendations.\n'
        '3. Give 2-3 physical activity recommendations.\n'
        '4. Note any behavioral interventions.\n'
        'Be concise and evidence-based.'
    )

    gemini = genai.GenerativeModel('gemini-1.5-flash')
    response = gemini.generate_content(prompt)
    return {'summary': cluster_summary, 'retrieved': retrieved, 'recommendation': response.text}


# 단일 클러스터로 테스트
test_cluster = cluster_summaries[0]
print(f'테스트 클러스터: {test_cluster["summary"][:120]}...')

In [ ]:
result = generate_recommendation(test_cluster['summary'], top_k=5)

print('=' * 60)
print(f'[Cluster {test_cluster["cluster"]}] 추천 결과')
print('=' * 60)
print(result['recommendation'])
print('\n검색된 청크:')
for i, r in enumerate(result['retrieved']):
    print(f'  [{i+1}] (score={r["score"]:.3f}) {r["chunk"][:80]}...')

## 전체 클러스터 추천 생성

In [ ]:
import time

all_results = []
for cs in cluster_summaries:
    print(f'\nCluster {cs["cluster"]} ({cs["dominant_label"]}) 처리 중...')
    try:
        res = generate_recommendation(cs['summary'], top_k=5)
        all_results.append({'cluster': cs['cluster'],
                            'label': cs['dominant_label'],
                            'recommendation': res['recommendation']})
        print(res['recommendation'][:300])
    except Exception as e:
        print(f'  오류: {e}')
    time.sleep(1)  # API rate limit 방지

print(f'\n완료: {len(all_results)}/{len(cluster_summaries)} 클러스터')

## BERTScore 평가

생성된 추천과 검색된 청크 간 의미론적 유사도를 BERTScore로 측정합니다.

In [ ]:
from bert_score import score as bert_score

if all_results:
    candidates = [r['recommendation'] for r in all_results]
    # 각 클러스터의 검색된 청크 재활용 (참조)
    references = [generate_recommendation(cluster_summaries[i]['summary'], top_k=3)['retrieved'][0]['chunk']
                  for i in range(len(all_results))]

    P, R, F1 = bert_score(
        candidates, references,
        lang='en', model_type='distilbert-base-uncased', verbose=False
    )
    print(f'BERTScore – Precision: {P.mean():.4f}, Recall: {R.mean():.4f}, F1: {F1.mean():.4f}')
    for i, (p, r, f) in enumerate(zip(P, R, F1)):
        print(f'  Cluster {all_results[i]["cluster"]}: P={p:.3f} R={r:.3f} F1={f:.3f}')
else:
    print('추천 결과 없음 – 이전 셀을 먼저 실행하세요.')